In [ ]:
import subprocess
import time

from paddleocr import PaddleOCR

import numpy as np

import cv2

import re

import os

from datetime import datetime

import traceback

class ADB: 
    def __init__(self, device_ip="127.0.0.1", port=5555):
        a = 1

        self.ocr = PaddleOCR(lang="korean", use_gpu=True, show_log=False)




    def get_ocr(self, file_name="capture.png", x_min=0, x_max=480, y_min=0, y_max=1000, y_threshold=10):

        img_path = f"C:\\Users\\NEC\\Pictures\\BlueStacks\\{file_name}"
        
        # 이미지 로드 및 크롭
        image = cv2.imread(img_path)  # 이미지를 읽음
        if image is None:
            raise FileNotFoundError(f"Image not found at path: {img_path}")
        
        # 관심 영역(ROI)만 크롭
        height, width, _ = image.shape
        cropped_image = image[y_min:y_max, x_min:x_max]  # y_min과 y_max로 이미지 크롭
        
        # 크롭된 이미지를 OCR 처리
        cropped_img_path = f"C:\\Users\\NEC\\Pictures\\BlueStacks\\cropped_{file_name}"
        cv2.imwrite(cropped_img_path, cropped_image)  # 임시 파일로 저장
        result = self.ocr.ocr(cropped_img_path, cls=False)

        # print(result)

        if result == [None] : # ocr 텍스트 없음조건
            return [None]

        # 1. 결과를 Y좌표 기준으로 필터링
        lines = []
        for line in result[0]:
            coords = line[0]  # OCR 좌표
            text = line[1][0]  # OCR 텍스트
            x_mean = np.mean([coords[0][0], coords[2][0]])  # X좌표 평균값
            y_mean = np.mean([coords[0][1], coords[2][1]])  # Y좌표 평균값

            # Y좌표 필터링
            lines.append({"x_mean": x_mean, "y_mean": y_mean, "text": text})

        # 2. Y좌표 기준으로 정렬
        lines.sort(key=lambda x: x["y_mean"])

        # 3. Y좌표 차이에 따라 그룹화
        grouped_lines = []
        current_line = []

        for item in lines:
            if not current_line:
                current_line.append(item)
            else:
                # 이전 Y좌표와 비교하여 같은 줄인지 판단
                if abs(item["y_mean"] - current_line[-1]["y_mean"]) <= y_threshold:
                    current_line.append(item)
                else:
                    # X좌표 기준으로 정렬 후 저장
                    grouped_lines.append(sorted(current_line, key=lambda x: x["x_mean"]))
                    current_line = [item]

        # 마지막 줄 추가
        if current_line:
            grouped_lines.append(sorted(current_line, key=lambda x: x["x_mean"]))

        # 4. 결과 출력
        # for line in grouped_lines:
            # print(" ".join([item["text"] for item in line]))

        return grouped_lines
    

    def get_ocr_raw(self, file_name="capture.png", x_min=0, x_max=480, y_min=0, y_max=1000, y_threshold=10, scale=3):

        img_path = f"C:\\Users\\NEC\\Pictures\\BlueStacks\\{file_name}"
        
        # 이미지 로드 및 크롭
        image = cv2.imread(img_path)  # 이미지를 읽음
        if image is None:
            raise FileNotFoundError(f"Image not found at path: {img_path}")
        
        # 관심 영역(ROI)만 크롭
        cropped_image = image[y_min:y_max, x_min:x_max]
        
        # 작은 영역일수록 확대해서 OCR 정확도 향상 (야외 등 작은 글자 인식용)
        if scale and scale != 1:
            h, w = cropped_image.shape[:2]
            cropped_image = cv2.resize(cropped_image, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)
            # 대비 강화 (흐린 글자 인식 개선)
            gray = cv2.cvtColor(cropped_image, cv2.COLOR_BGR2GRAY)
            cropped_image = cv2.cvtColor(cv2.equalizeHist(gray), cv2.COLOR_GRAY2BGR)
        
        cropped_img_path = f"C:\\Users\\NEC\\Pictures\\BlueStacks\\cropped_{file_name}"
        cv2.imwrite(cropped_img_path, cropped_image)
        result = self.ocr.ocr(cropped_img_path, cls=False)

        return result
    

    def process_ocr(self, result, x_min=0, x_max=480, y_min=0, y_max=1000, y_threshold=10, scale=1, merge=True):
        if result is None or result == [None] or result[0] is None:
            return []
        processed_lines = []

        for block in result[0]:
            coords = block[0]
            text = block[1][0]
            confidence = block[1][1]  # 인식률

            # 업스케일된 이미지로 OCR한 경우: scale로 나눈 뒤 원본 기준으로 보정
            x_coords = [point[0] / scale + x_min for point in coords]
            y_coords = [point[1] / scale + y_min for point in coords]

            x_mean = np.mean(x_coords)
            y_mean = np.mean(y_coords)

            # 각 영역의 x 최소/최대와 y 최소/최대 계산
            x_left = min(x_coords)
            x_right = max(x_coords)
            y_top = min(y_coords)
            y_bottom = max(y_coords)

            processed_lines.append({
                "text": text,
                "x_mean": x_mean,
                "y_mean": y_mean,
                "x_left": x_left,
                "x_right": x_right,
                "y_top": y_top,
                "y_bottom": y_bottom,
                "confidence": confidence
            })

        # Y좌표 기준으로 정렬
        processed_lines.sort(key=lambda x: x["y_mean"])

        # mod=False: 병합 없이 블록 단위로 반환
        if not merge:
            return [[item["text"], item["x_mean"], item["y_mean"],
                     item["x_left"], item["x_right"], item["y_top"], item["y_bottom"],
                     item["confidence"]] for item in processed_lines]

        # 같은 줄 텍스트 병합 (mod=True, 기본)
        grouped_lines = []
        current_line = []

        for item in processed_lines:
            if not current_line:
                current_line.append(item)
            else:
                # 같은 줄인지 확인 (y_threshold 기준)
                if abs(item["y_mean"] - current_line[-1]["y_mean"]) <= y_threshold:
                    current_line.append(item)
                else:
                    # 병합 후 저장 (X좌표 기준으로 정렬)
                    grouped_lines.append(sorted(current_line, key=lambda x: x["x_mean"]))
                    current_line = [item]

        if current_line:
            grouped_lines.append(sorted(current_line, key=lambda x: x["x_mean"]))

        # 병합된 결과 처리
        final_output = []
        for group in grouped_lines:
            merged_text = " ".join([item["text"] for item in group])
            x_mean = np.mean([item["x_mean"] for item in group])
            y_mean = np.mean([item["y_mean"] for item in group])
            x_left = min([item["x_left"] for item in group])
            x_right = max([item["x_right"] for item in group])
            y_top = min([item["y_top"] for item in group])
            y_bottom = max([item["y_bottom"] for item in group])
            confidence = np.mean([item["confidence"] for item in group])

            final_output.append([merged_text, x_mean, y_mean, x_left, x_right, y_top, y_bottom, confidence])

        return final_output


    def ocr_to_plain(self, ocr) :

        result_text = ""

        for line in ocr:
            # 각 줄의 텍스트를 공백으로 연결하고 줄바꿈 추가
            line_text = " ".join([item["text"] for item in line])
            result_text += line_text + "\n"

        # 마지막 줄바꿈 제거
        result_text = result_text.strip()

        return result_text    


    def has_keywords(processed_result, keywords, min_count=1):
        """
        Checks if at least `min_count` of the given `keywords` are present (as substrings)
        in the first element of each sublist in `processed_result`.

        Args:
            processed_result (list): The OCR result list.
            keywords (list or str): Keywords to check for.
            min_count (int): Minimum number of keywords that must be found.

        Returns:
            bool: True if at least `min_count` keywords are found, False otherwise.
        """
        # Ensure keywords is a list
        if isinstance(keywords, str):
            keywords = [keywords]
        found = set()
        for keyword in keywords:
            for item in processed_result:
                if keyword in str(item[0]):
                    found.add(keyword)
                    break  # No need to count the same keyword more than once
        return len(found) >= min_count


        
            

    

In [10]:
adb = ADB(port=5555)

In [13]:
result = adb.get_ocr_raw(file_name="test2.png", x_min=0, x_max=540, y_min=870, y_max=960, y_threshold=10, scale=1)
processed_result = adb.process_ocr(result=result, x_min=0, x_max=540, y_min=870, y_max=960, y_threshold=10, scale=1, merge=False)

processed_result

[['토벌', 53.0, 941.0, 37.0, 69.0, 932.0, 950.0, 0.9983649253845215],
 ['가방', 229.0, 941.0, 212.0, 246.0, 932.0, 950.0, 0.9989116191864014],
 ['영웅', 142.5, 941.5, 129.0, 156.0, 934.0, 949.0, 0.9975530505180359],
 ['상점', 314.0, 941.5, 299.0, 329.0, 932.0, 951.0, 0.99821937084198],
 ['연맹', 401.0, 941.5, 385.0, 417.0, 932.0, 951.0, 0.9982953667640686],
 ['야외', 486.0, 942.0, 469.0, 503.0, 931.0, 953.0, 0.9998219013214111]]

In [ ]:
result = adb.get_ocr_raw(
    file_name="capture_barracks_button.png",
    x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max,
    y_threshold=y_threshold, scale=scale
)
processed_result = adb.process_ocr(
    result=result, x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max,
    y_threshold=y_threshold, scale=scale
)

In [ ]:

def check_msg(msg, x_min, x_max, y_min, y_max, y_threshold, scale):
    self.screen_shot(name="_barracks_button")
    result = self.get_ocr_raw(
        file_name="capture_barracks_button.png",
        x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max,
        y_threshold=y_threshold, scale=scale
    )
    processed_result = self.process_ocr(
        result=result, x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max,
        y_threshold=y_threshold, scale=scale
    )
    return any(msg in str(item[0]) for item in processed_result)

In [2]:


import subprocess
import time


# 특정 시간이 되면 실행

from datetime import datetime

# while datetime.now().hour < 23 and datetime.now().hour > 10 :
#     time.sleep(60)

print("시작!")


commands = [
    [r"C:\\Program Files\BlueStacks_nxt\\HD-Player.exe", "--instance", "Pie64_1"], # 5555
    # [r"C:\\Program Files\BlueStacks_nxt\\HD-Player.exe", "--instance", "Pie64_12"],
    # [r"C:\\Program Files\BlueStacks_nxt\\HD-Player.exe", "--instance", "Pie64_13"],
    # [r"C:\\Program Files\BlueStacks_nxt\\HD-Player.exe", "--instance", "Pie64_14"],
    # [r"C:\\Program Files\BlueStacks_nxt\\HD-Player.exe", "--instance", "Pie64_7"],
    # [r"C:\\Program Files\BlueStacks_nxt\\HD-Player.exe", "--instance", "Pie64_8"],
    # [r"C:\\Program Files\BlueStacks_nxt\\HD-Player.exe", "--instance", "Pie64_9"],
    # [r"C:\\Program Files\BlueStacks_nxt\\HD-Player.exe", "--instance", "Pie64_10"],
    # [r"C:\\Program Files\BlueStacks_nxt\\HD-Player.exe", "--instance", "Pie64_11"],
]


processes = []

for cmd in commands:
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    processes.append(process)
    time.sleep(3)  # 명령 실행 간 지연 시간 추가

time.sleep(10)


# ADB 연결

import time

adbs = []

adbs.append(ADB(port=5555))
# adbs.append(ADB(port=5675))
# adbs.append(ADB(port=5685))
# adbs.append(ADB(port=5695))
# adbs.append(ADB(port=5625))
# adbs.append(ADB(port=5635))
# adbs.append(ADB(port=5645))
# adbs.append(ADB(port=5655))
# adbs.append(ADB(port=5665))


# adbs.append(ADB(port=5675))

for adb in adbs :
    adb.connect()

for adb in adbs :
    adb.home()

for adb in adbs :
    adb.start_kingshot()





시작!


In [ ]:
timer = time.time() - 0 * 60 * 60 # 철수 원하면 3으로
back_timer = time.time()
back_trigger = False

while True:

    # reconnet 창이 떠있는지 확인
    reconnect_check = adbs[0].check_reconnect()
    reconnect_check 


    if reconnect_check == False :

        # 유닛 생산
        for adb in adbs :
            adb.unit_training()
            time.sleep(1)


        # 건물 업그레이드
        for adb in adbs :
            adb.build_city()
            time.sleep(1)
        

    # 2시간마다 자원부대 철수
    if time.time() - timer > 2.5 * 60 * 60:

        if reconnect_check == True :
            adb.tap(380,595) # reconnect
            time.sleep(10)

        # 현재 위치 판단
        adb.screen_shot(name="_inout")
        adb.crop_image(file_name="capture_inout.png", x_min=465, x_max=505, y_min=930, y_max=955)
        result = adb.compare_inout(cropped_file_name="cropped_capture_inout.png")  # "in" 또는 "out"

        if result == "out" : # out 버튼이 뜨는 경우 (도시 안에 있는 경우)
            adb.tap(490,910) # 야외로 나가기
            time.sleep(5)

        adb.troops_back()
        time.sleep(1)

        adb.tap(490,910) # 도시로 돌아가기
        time.sleep(5)

        timer = time.time()
        back_timer = time.time()
        back_trigger = True



    # 자원부대 철수 후 5분 지나면 다시 자원 채취
    elif time.time() - back_timer > 5 * 60 and back_trigger :

        if reconnect_check == True :
            adb.tap(380,595) # reconnect
            time.sleep(10)

        # 현재 위치 판단
        adb.screen_shot(name="_inout")
        adb.crop_image(file_name="capture_inout.png", x_min=465, x_max=505, y_min=930, y_max=955)
        result = adb.compare_inout(cropped_file_name="cropped_capture_inout.png")  # "in" 또는 "out"

        if result == "out" : # out 버튼이 뜨는 경우 (도시 안에 있는 경우)
            adb.tap(490,910) # 야외로 나가기
            time.sleep(5)

        adb.resource_farming()
        time.sleep(1)

        adb.tap(490,910) # 도시로 돌아가기
        time.sleep(5)

        timer = time.time()
        back_trigger = False


    time.sleep(60)



None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None


In [ ]:
from numpy._typing import _128Bit


def build_city(adb) :

    def check_upgrade(adb) :
        adb.screen_shot(name="_upgrade")
        adb.crop_image(file_name="capture_upgrade.png", x_min=350, x_max=440, y_min=880, y_max=905)
        return adb.matches_reference(
            cropped_file_name="cropped_capture_upgrade.png",
            reference_name="upgrade.png",
            mse_threshold=0.800
        )

    def upgrade_ocr(adb) :

        adb.screen_shot(name="_upgrade_button")
        result = adb.get_ocr_raw(file_name="capture_upgrade_button.png", x_min=360, x_max=535, y_min=350, y_max=655, y_threshold=10, scale=3)
        processed_result = adb.process_ocr(result=result, x_min=360, x_max=535, y_min=350, y_max=655, y_threshold=10, scale=3)

        upgrade_items = [item for item in processed_result if item[0] == '업그레이드']
        first_upgrade = upgrade_items[0] if upgrade_items else None
        return first_upgrade

    # adb.tap(10,140)
    # time.sleep(0.5)
    # adb.tap(10,140)
    # time.sleep(0.5)

    adb.tap(10,415)
    time.sleep(0.5)
    adb.tap(305,285) # 건물 1
    time.sleep(3)

    adb.tap(80,500) # 손가락 커서 치우기
    adb.tap(80,500) # 손가락 커서 치우기
    time.sleep(3)
    result = upgrade_ocr(adb)
    # print(result)

    if result != None :
        x = result[1]
        y = result[2]

        adb.tap(x,y) # 업그레이드 버튼
        time.sleep(0.5)
        adb.tap(495,900) # 업그레이드 확인
        time.sleep(3)
        adb.tap(270,330) # 도움 버튼

    adb.tap(10,140)
    time.sleep(0.5)
    adb.tap(10,140)
    time.sleep(0.5)
    time.sleep(1)

    adb.tap(10,415)
    time.sleep(0.5)
    adb.tap(305,335) # 건물 2
    time.sleep(3)

    adb.tap(80,500) # 손가락 커서 치우기
    adb.tap(80,500) # 손가락 커서 치우기
    time.sleep(3)
    result = upgrade_ocr(adb)
    print(result)

    if result != None :
        x = result[1]
        y = result[2]

        adb.tap(x,y) # 업그레이드 버튼
        time.sleep(0.5)
        adb.tap(495,900) # 업그레이드 확인
        time.sleep(3)
        adb.tap(270,330) # 도움 버튼


build_city(adbs[0])

None
try 2
None


In [28]:
def upgrade_ocr(adb) :

    adb.screen_shot(name="_upgrade_button")
    result = adb.get_ocr_raw(file_name="capture_upgrade_button.png", x_min=360, x_max=535, y_min=350, y_max=655, y_threshold=10, scale=3)
    processed_result = adb.process_ocr(result=result, x_min=360, x_max=535, y_min=350, y_max=655, y_threshold=10, scale=3)

    upgrade_items = [item for item in processed_result if item[0] == '업그레이드']
    first_upgrade = upgrade_items[0] if upgrade_items else None
    first_upgrade

r = upgrade_ocr(adbs[0])

r

In [ ]:
x = 0
y = 0

adbs[0].screen_shot(name="_upgrade_check")
result = adbs[0].get_ocr_raw(file_name="capture_upgrade_check.png", x_min=110, x_max=465, y_min=470, y_max=680, y_threshold=10, scale=3)
processed_result = adbs[0].process_ocr(result=result, x_min=110, x_max=465, y_min=470, y_max=680, y_threshold=10, scale=3)

has_detail = any("상세" in str(item[0]) for item in processed_result)
has_upgrade = any("업그레이드" in str(item[0]) for item in processed_result)
result = has_detail and has_upgrade

if result == True :
    upgrade_items = [item for item in processed_result if "업그레이드" in str(item[0])]
    first_upgrade = upgrade_items[0] if upgrade_items else None
    x = first_upgrade[1]
    y = first_upgrade[2]
    
return result, x, y-50

269.83333333333337 615.3333333333334


In [ ]:

upgrade_items = [item for item in processed_result if item[0] == '업그레이드']
first_upgrade = upgrade_items[0] if upgrade_items else None
return first_upgrade

In [ ]:
adbs[0].screen_shot(name="_upgrade")
adbs[0].crop_image(file_name="capture_upgrade.png", x_min=350, x_max=440, y_min=880, y_max=905)

mse_threshold = 0.800
result = adbs[0].matches_reference(
    cropped_file_name="cropped_capture_upgrade.png",
    reference_name="upgrade.png",
    mse_threshold=mse_threshold
)

result

False

In [10]:
image = adbs[0].screen_shot(name="_upgrade")
cropped_image = adbs[0].crop_image(file_name="capture_upgrade.png", x_min=350, x_max=440, y_min=880, y_max=905)

In [6]:
adbs[0].screen_shot(name="_inout")
adbs[0].crop_image(file_name="capture_inout.png", x_min=465, x_max=505, y_min=930, y_max=955)
result = adbs[0].compare_inout(cropped_file_name="cropped_capture_inout.png")  # "in" 또는 "out"


result

'in'

In [7]:
adbs[0].screen_shot(name="_upgrade_button")
result = adbs[0].get_ocr_raw(file_name="capture_upgrade_button.png", x_min=360, x_max=535, y_min=350, y_max=655, y_threshold=10, scale=3)
processed_result = adbs[0].process_ocr(result=result, x_min=360, x_max=535, y_min=350, y_max=655, y_threshold=10, scale=3)

upgrade_items = [item for item in processed_result if item[0] == '업그레이드']
first_upgrade = upgrade_items[0] if upgrade_items else None
first_upgrade

In [21]:
processed_result

[['업그레이드', 655.5, 494.5, 528.0, 783.0, 464.0, 525.0, 0.9997023344039917],
 ['업그레이드', 653.5, 759.0, 530.0, 777.0, 732.0, 786.0, 0.9997893571853638],
 ['을', 394.5, 804.0, 370.0, 419.0, 776.0, 832.0, 0.9993982315063477],
 ['6OM/S8K', 696.5, 839.5, 569.0, 824.0, 812.0, 867.0, 0.9600241780281067]]

In [4]:
adbs[0].resource_farming()
time.sleep(1)


In [ ]:
def back(adb) :
    adb.tap(490,910) # 야외로 나가기
    time.sleep(10)

    adb.troops_back()
    timer = time.time()

    adb.tap(35,660) # 서치버튼
    time.sleep(0.5)
    adb.tap(305,685) # 빵
    time.sleep(0.5)
    adb.tap(270,910) # 검색
    time.sleep(3)
    adb.tap(270,470) # 채집
    time.sleep(0.5)
    adb.tap(320,190) # 영웅 2 취소
    time.sleep(0.5)
    adb.tap(460,190) # 영웅 3 취소
    time.sleep(0.5)
    adb.tap(220,890) # 균등배치
    time.sleep(0.5)
    adb.tap(410,910) # 출정
    time.sleep(0.5)


    adb.tap(35,660) # 서치버튼
    time.sleep(0.5)
    adb.tap(385,685) # 목재
    time.sleep(0.5)
    adb.tap(270,910) # 검색
    time.sleep(3)
    adb.tap(270,470) # 채집
    time.sleep(0.5)
    adb.tap(320,190) # 영웅 2 취소
    time.sleep(0.5)
    adb.tap(460,190) # 영웅 3 취소
    time.sleep(0.5)
    adb.tap(220,890) # 균등배치
    time.sleep(0.5)
    adb.tap(410,910) # 출정
    time.sleep(0.5)


    adb.tap(35,660) # 서치버튼
    time.sleep(0.5)
    adb.tap(476,685) # 철광
    time.sleep(0.5)
    adb.tap(270,910) # 검색
    time.sleep(3)
    adb.tap(270,470) # 채집
    time.sleep(0.5)
    adb.tap(320,190) # 영웅 2 취소
    time.sleep(0.5)
    adb.tap(460,190) # 영웅 3 취소
    time.sleep(0.5)
    adb.tap(220,890) # 균등배치
    time.sleep(0.5)
    adb.tap(410,910) # 출정
    time.sleep(0.5)





back(adbs[0])


In [ ]:
def back(adb) :

    

back(adbs[0])

KeyboardInterrupt: 

In [4]:
def back(adb) :

    adb.tap(165,200)
    time.sleep(0.5)
    adb.tap(385,590)
    time.sleep(0.5)


    adb.tap(165,245)
    time.sleep(0.5)
    adb.tap(385,590)
    time.sleep(0.5)


    adb.tap(165,290)
    time.sleep(0.5)
    adb.tap(385,590)
    time.sleep(0.5)


    adb.tap(165,335)
    time.sleep(0.5)
    adb.tap(385,590)
    time.sleep(0.5)



back(adbs[0])


In [ ]:
for adb in adbs :

    

In [ ]:

adbs[0].state

'C:\\platform-tools-latest-windows\\platform-tools\\adb.exe'

In [22]:
def build_city(adb) :

    def check_upgrade(adb) :
        adb.screen_shot(name="_upgrade")
        adb.crop_image(file_name="capture_upgrade.png", x_min=350, x_max=440, y_min=880, y_max=905)
        return adb.matches_reference(
            cropped_file_name="cropped_capture_upgrade.png",
            reference_name="upgrade.png",
            mse_threshold=0.800
        )

    def upgrade_ocr(adb, x_min, x_max, y_min, y_max, y_threshold, scale) :

        adb.screen_shot(name="_upgrade_button")
        result = adb.get_ocr_raw(file_name="capture_upgrade_button.png", x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, y_threshold=y_threshold, scale=scale)
        processed_result = adb.process_ocr(result=result, x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, y_threshold=y_threshold, scale=scale)

        upgrade_items = [item for item in processed_result if item[0] == '업그레이드']
        first_upgrade = upgrade_items[0] if upgrade_items else None
        return first_upgrade

    def upgrade_check(adb) :

        x = 0
        y = 0

        adb.screen_shot(name="_upgrade_check")
        result = adb.get_ocr_raw(file_name="capture_upgrade_check.png", x_min=110, x_max=465, y_min=470, y_max=680, y_threshold=10, scale=3)
        processed_result = adb.process_ocr(result=result, x_min=110, x_max=465, y_min=470, y_max=680, y_threshold=10, scale=3)

        has_detail = any("상세" in str(item[0]) for item in processed_result)
        has_upgrade = any("업그레이드" in str(item[0]) for item in processed_result)
        result = has_detail and has_upgrade

        if result == True :
            upgrade_items = [item for item in processed_result if "업그레이드" in str(item[0])]
            first_upgrade = upgrade_items[0] if upgrade_items else None
            x = first_upgrade[1]
            y = first_upgrade[2]
            
        return result, x, y-50


    adb.tap(10,415)
    time.sleep(0.5)
    adb.tap(305,285) # 건물 1
    time.sleep(3)

    result, x, y = upgrade_check(adb)
    if result == True :
        adb.tap(x,y) # 업그레이드 버튼 누르기
        time.sleep(0.5)
        result = upgrade_ocr(adb, x_min=240, x_max=510, y_min=580, y_max=820, y_threshold=10, scale=3)
        print(result)
        x = result [1]
        y = result [2]
        adb.tap(x,y) # 업그레이드 버튼 누르기
        time.sleep(3)
        adb.tap(270,330) # 도움 버튼

    adb.tap(80,500) # 손가락 커서 치우기
    adb.tap(80,500) # 손가락 커서 치우기
    time.sleep(3)
    result = upgrade_ocr(adb, x_min=360, x_max=535, y_min=350, y_max=655, y_threshold=10, scale=3)
    print(result)

    if result != None :
        x = result[1]
        y = result[2]

        adb.tap(x,y) # 업그레이드 버튼
        time.sleep(0.5)
        adb.tap(495,900) # 업그레이드 확인
        time.sleep(3)
        adb.tap(270,330) # 도움 버튼

    # 1스텝 종료

    adb.tap(10,140)
    time.sleep(0.5)
    adb.tap(10,140)
    time.sleep(0.5)
    time.sleep(1)

    adb.tap(10,415)
    time.sleep(0.5)
    adb.tap(305,335) # 건물 2
    time.sleep(3)

    result, x, y = upgrade_check(adb)
    if result == True :
        adb.tap(x,y) # 업그레이드 버튼 누르기
        time.sleep(0.5)
        result = upgrade_ocr(adb, x_min=240, x_max=510, y_min=580, y_max=820, y_threshold=10, scale=3)
        x = result [1]
        y = result [2]
        adb.tap(x,y) # 업그레이드 버튼 누르기
        time.sleep(3)
        adb.tap(270,330) # 도움 버튼

    adb.tap(80,500) # 손가락 커서 치우기
    adb.tap(80,500) # 손가락 커서 치우기
    time.sleep(3)
    result = upgrade_ocr(adb, x_min=360, x_max=535, y_min=350, y_max=655, y_threshold=10, scale=3)
    print(result)

    if result != None :
        x = result[1]
        y = result[2]

        adb.tap(x,y) # 업그레이드 버튼
        time.sleep(0.5)
        adb.tap(495,900) # 업그레이드 확인
        time.sleep(3)
        adb.tap(270,330) # 도움 버튼


build_city(adb=adbs[0])







None
None
